# 5b_GBM_TARGETS

Apply trained GBM quantile models to Antarctic and Greenland target grids.

Requires five separate pkl artefacts (one per quantile) from `4b_GBM_MODEL`:
`gbm_q05_model.pkl`, `gbm_q25_model.pkl`, `gbm_q50_model.pkl`,
`gbm_q75_model.pkl`, `gbm_q95_model.pkl`.

**Variables written:** identical structure to 5a but prefixed `gbm_`.

**Spline:** same Q50-only correction, offset propagated to other quantiles.

**Files written:**
- `output/targets/Aq1_5_GBM_v{MODEL_VERSION}.nc` — Antarctica
- `output/targets/Kq1_5_GBM_v{MODEL_VERSION}.nc` — Greenland

## 0. Config patch check

In [ ]:
# ── config.py additions needed ───────────────────────────────────────────────
# Add these lines to config.py if not already present:
#
# MODEL_VERSION = '0_2'
#
# model_paths update — add Q25 and Q75 GBM:
# model_paths = {
#     'qrf'       : model_dir / 'qrf_model.pkl',
#     'gbm_q05'   : model_dir / 'gbm_q05_model.pkl',
#     'gbm_q25'   : model_dir / 'gbm_q25_model.pkl',
#     'gbm_q50'   : model_dir / 'gbm_q50_model.pkl',
#     'gbm_q75'   : model_dir / 'gbm_q75_model.pkl',
#     'gbm_q95'   : model_dir / 'gbm_q95_model.pkl',
#     'sim_correction': model_dir / 'sim_correction_spline.pkl',
#     'model_metrics' : model_dir / 'model_metrics.csv',
# }
#
# Each GBM artefact pkl is produced by 4b_GBM_MODEL for each quantile.
# The pkl must contain keys: 'model','scaler','spline','obs_sel','PARAMS'
print('Config note displayed.')


## 1. Imports & constants

In [ ]:
import sys, json, pickle, warnings, datetime
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
from config import (
    obs_model, q_clip_min, q_clip_max, random_state,
    ant_crs, grl_crs, model_dir, targets_dir, param_paths,
    netcdf_compression_level, NETCDF_AUTHOR, NETCDF_CONVENTIONS,
    PRED_QUANTILES, TARGET_GRIDS, MODEL_VERSION,
)
targets_dir.mkdir(parents=True, exist_ok=True)
QUANTILES = PRED_QUANTILES   # [0.05, 0.25, 0.50, 0.75, 0.95]
Q50_IDX   = QUANTILES.index(0.50)
print(f'quantiles    : {QUANTILES}')
print(f'model version: {MODEL_VERSION}')
print(f'targets_dir  : {targets_dir}')


## 2. Helpers

In [ ]:
# ── Grid loader ───────────────────────────────────────────────────────────────
def load_grid(grid_cfg, obs_sel):
    label, xcol, ycol = grid_cfg['label'], grid_cfg['x_col'], grid_cfg['y_col']
    df = pd.read_parquet(grid_cfg['parquet'])
    print(f'\n[{label.upper()}] {len(df):,} rows')
    missing = [f for f in obs_sel if f not in df.columns]
    if missing:
        warnings.warn(f'Missing features {missing} — filling NaN')
        for f in missing: df[f] = np.nan
    nan_cols = {f: int(df[f].isna().sum()) for f in obs_sel if df[f].isna().any()}
    if nan_cols: print(f'  NaN counts: {nan_cols}')
    else: print(f'  All {len(obs_sel)} features OK, no NaNs')
    x_vals = np.sort(df[xcol].unique()).astype(np.float64)
    y_vals = np.sort(df[ycol].unique()).astype(np.float64)
    nx, ny = len(x_vals), len(y_vals)
    is_reg = (len(df) == nx * ny)
    if is_reg:
        dx, dy = np.diff(x_vals), np.diff(y_vals)
        print(f'  Regular {ny}x{nx}  dx={dx[0]:.0f}m dy={dy[0]:.0f}m')
    else:
        print(f'  Irregular {len(df):,} pts (expected {ny*nx:,}) — griddata fallback')
    return df, x_vals, y_vals, ny, nx, is_reg

def build_X(df, obs_sel, scaler):
    X_raw  = df[obs_sel].values.astype(np.float32)
    finite = np.isfinite(X_raw).all(axis=1)
    X_sc   = np.full_like(X_raw, np.nan)
    if finite.any():
        X_sc[finite] = scaler.transform(X_raw[finite]).astype(np.float32)
    print(f'  finite points: {finite.sum():,} / {len(df):,}')
    return X_sc, finite

def to_2d(vals_1d, df, x_vals, y_vals, xcol, ycol, is_reg):
    ny, nx = len(y_vals), len(x_vals)
    if is_reg:
        tmp = df[[xcol, ycol]].copy()
        tmp['_v'] = vals_1d
        return tmp.pivot(index=ycol, columns=xcol, values='_v').values.astype(np.float32)
    from scipy.interpolate import griddata
    gx, gy = np.meshgrid(x_vals, y_vals)
    return griddata(df[[xcol, ycol]].values, vals_1d, (gx, gy), method='nearest').astype(np.float32)

def apply_spline(spline, vals):
    out = np.full(vals.shape, np.nan, dtype=np.float32)
    ok  = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), q_clip_min, q_clip_max)
    return out

def shannon_H_norm(values_2d, bin_width=0.010):
    v = values_2d.ravel()
    v = v[np.isfinite(v)]
    if len(v) == 0: return np.nan
    bins = np.arange(q_clip_min, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(v, bins=bins)
    p = counts / counts.sum()
    p = p[p > 0]
    H = float(scipy_entropy(p))
    H_max = float(np.log(len(bins) - 1))
    return round(H / H_max, 6) if H_max > 0 else 0.0

def make_ds(arrays, x_vals, y_vals, grid_cfg, params, obs_sel, model_tag, extra=None):
    coords = {'y': y_vals, 'x': x_vals}
    dvars = {name: xr.DataArray(arr, dims=['y','x'], coords=coords,
                                attrs={'units':'W m-2','long_name':name.replace('_',' ')})
             for name, arr in arrays.items()}
    ds = xr.Dataset(dvars, coords=coords)
    attrs = dict(
        title=f'{model_tag} heat-flow prediction',
        model=model_tag, version=MODEL_VERSION,
        institution=NETCDF_AUTHOR, Conventions=NETCDF_CONVENTIONS,
        crs=grid_cfg['crs'], epsg=str(grid_cfg['epsg']),
        region=grid_cfg['label'].upper(),
        created=datetime.datetime.utcnow().isoformat()+'Z',
        obs_sel=json.dumps(obs_sel),
        quantiles=json.dumps(QUANTILES),
        q_clip_min=float(q_clip_min), q_clip_max=float(q_clip_max),
        params_json=json.dumps(params, default=str),
    )
    if extra: attrs.update(extra)
    ds.attrs = attrs
    return ds

def save_nc(ds, path):
    enc = {v: {'zlib':True,'complevel':netcdf_compression_level,'dtype':'float32'}
           for v in ds.data_vars}
    ds.to_netcdf(path, encoding=enc)
    print(f'  Saved {path}  ({path.stat().st_size/1e6:.1f} MB)')

print('Helpers ready.')


## 3. Load GBM artefacts (all 5 quantiles)

In [ ]:
# Load all 5 quantile GBM models
gbm_models = {}
qnames     = ['q05','q25','q50','q75','q95']
qkeys      = ['gbm_q05','gbm_q25','gbm_q50','gbm_q75','gbm_q95']

for qk, qn in zip(qkeys, qnames):
    pkl_path = model_dir / f'{qk}_model.pkl'
    with open(pkl_path, 'rb') as fp:
        art = pickle.load(fp)
    gbm_models[qn] = art
    print(f'  Loaded {qk}: {type(art["model"]).__name__}')

# All GBM models share the same scaler and obs_sel (from Q50)
scaler   = gbm_models['q50']['scaler']
spline   = gbm_models['q50']['spline']
obs_sel  = gbm_models['q50']['obs_sel']
PARAMS   = gbm_models['q50']['PARAMS']
print(f'\nGBM obs_sel ({len(obs_sel)}): {obs_sel}')


## 4. Predict & write netCDF

Five sequential `model.predict()` calls per region (one per quantile).
Each call is a single forward pass — fast (~10–30s each).

In [ ]:
import time
results_gbm = {}

for grid_cfg in TARGET_GRIDS:
    label = grid_cfg['label']
    prefix = 'Aq1_5' if label == 'ant' else 'Kq1_5'
    out_path = targets_dir / f'{prefix}_GBM_v{MODEL_VERSION}.nc'

    df, x_vals, y_vals, ny, nx, is_reg = load_grid(grid_cfg, obs_sel)
    X_sc, finite = build_X(df, obs_sel, scaler)
    xcol, ycol = grid_cfg['x_col'], grid_cfg['y_col']

    arrays = {}
    q_preds_all = np.full((len(df), len(qnames)), np.nan, dtype=np.float32)

    for qi, qn in enumerate(qnames):
        t0 = time.time()
        mdl = gbm_models[qn]['model']
        pred = np.full(len(df), np.nan, dtype=np.float32)
        pred[finite] = np.clip(
            mdl.predict(X_sc[finite]).astype(np.float32),
            q_clip_min, q_clip_max
        )
        q_preds_all[:, qi] = pred
        print(f'  [{label.upper()}] GBM {qn} predict: {(time.time()-t0):.1f}s')

    # ── spline correction on Q50; offsets for other quantiles ─────────────
    q50_raw  = q_preds_all[:, 2]   # index 2 = q50
    q50_corr = apply_spline(spline, q50_raw)
    offset   = q50_corr - q50_raw

    for qi, qn in enumerate(qnames):
        raw_1d  = q_preds_all[:, qi]
        corr_1d = np.clip(raw_1d + offset, q_clip_min, q_clip_max)
        arrays[f'gbm_{qn}_raw' ] = to_2d(raw_1d,  df, x_vals, y_vals, xcol, ycol, is_reg)
        arrays[f'gbm_{qn}_corr'] = to_2d(corr_1d, df, x_vals, y_vals, xcol, ycol, is_reg)

    # ── derived uncertainty metrics ───────────────────────────────────────
    iqr50_raw  = q_preds_all[:,3] - q_preds_all[:,1]
    iqr90_raw  = q_preds_all[:,4] - q_preds_all[:,0]
    sigma_raw  = iqr90_raw / (2 * 1.6449)
    iqr50_corr = np.clip(iqr50_raw, 0, q_clip_max)
    iqr90_corr = np.clip(iqr90_raw, 0, q_clip_max)
    sigma_corr = iqr90_corr / (2 * 1.6449)

    for name, arr1d in [
        ('gbm_iqr50_raw',  iqr50_raw),  ('gbm_iqr90_raw',  iqr90_raw),
        ('gbm_sigma_raw',  sigma_raw),
        ('gbm_iqr50_corr', iqr50_corr), ('gbm_iqr90_corr', iqr90_corr),
        ('gbm_sigma_corr', sigma_corr),
    ]:
        arrays[name] = to_2d(arr1d, df, x_vals, y_vals, xcol, ycol, is_reg)

    # ── Shannon entropy ───────────────────────────────────────────────────
    H_raw  = shannon_H_norm(arrays['gbm_q50_raw'])
    H_corr = shannon_H_norm(arrays['gbm_q50_corr'])
    print(f'  Shannon H (norm): raw={H_raw:.4f}  corr={H_corr:.4f}')

    # ── build & save dataset ──────────────────────────────────────────────
    ds = make_ds(arrays, x_vals, y_vals, grid_cfg, PARAMS, obs_sel,
                 model_tag=f'{prefix}_GBM',
                 extra={'shannon_H_q50_raw': H_raw,
                        'shannon_H_q50_corr': H_corr,
                        'spline_applied': 'Q50 only; other quantiles shifted by Q50 offset'})
    save_nc(ds, out_path)
    results_gbm[label] = ds
    print(f'  [{label.upper()}] done.')

print('\nAll GBM target grids complete.')


## 5. Summary statistics

In [ ]:
for label, ds in results_gbm.items():
    print(f'\n[{label.upper()}] variables:')
    for v in ds.data_vars:
        arr = ds[v].values
        fin = arr[np.isfinite(arr)]
        print(f'  {v:30s}  mean={fin.mean()*1e3:.1f}  '
              f'std={fin.std()*1e3:.1f}  '
              f'[{fin.min()*1e3:.1f}, {fin.max()*1e3:.1f}] mW/m2')
